# 06 — Reasoning-Oriented Prompting

## Scenario
A service is throwing 500 errors. We have the recent system logs. We need an AI to recommend a triage action. 

**The Danger:** Naive prompts often cause models to jump to conclusions (like "Restart the server") without checking the evidence. We need to force the model to compute observable reasoning steps before it answers.

## Step 1: The Naive Direct Prompt

We ask for the answer directly. Models trained on internet text often default to "turn it off and on again" for technical issues if they aren't forced to read the context carefully.

## Step 2: Structured Chain-of-Thought (CoT)

We can force the model to reason *before* it outputs the `recommended_action`. 
**State-of-the-Art Technique:** When using JSON/Structured outputs, we add a `reasoning_steps` array to the top of our Pydantic schema. Because LLMs generate tokens sequentially, forcing the JSON to output the reasoning array *first* guarantees the model computes its logic before committing to the final answer.

## Step 3: Compound System (Planner / Verifier)

For critical actions (like actually executing code), a single LLM call is too risky. We split the reasoning into a Compound AI System: one call plans/proposes, a separate call (with a different prompt) verifies the proposal against safety rules.

In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab06 import *

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Step 1: The Naive Direct Prompt

In [ ]:
request = next(r for r in build_requests() if r.case_id == "i06/naive/pool-exhaustion")
print("SYSTEM:\n", request.system)
print("USER:\n", request.messages[0].text)
response = client.generate(request)
print("RECORDED RESPONSE:\n", response.text)
parsed = TriageRecommendation.model_validate_json(response.text)
print("PARSED:", parsed)
assert parsed.recommended_action == "restart the application service"

## Step 2: Structured Chain-of-Thought (CoT)

In [ ]:
request = next(r for r in build_requests() if r.case_id == "i06/cot/pool-exhaustion")
print("SYSTEM:\n", request.system)
print("USER:\n", request.messages[0].text)
response = client.generate(request)
print("RECORDED RESPONSE:\n", response.text)
parsed = TriageRecommendation.model_validate_json(response.text)
print("PARSED:", parsed)
assert root_cause_named(parsed.reasoning_steps, "connection pool exhausted")

## Step 3: Compound System (Planner / Verifier)

In [ ]:
proposal = next(r for r in build_requests() if r.case_id == "i06/cot/pool-exhaustion")
verification_request = next(r for r in build_requests() if r.case_id == "i06/verifier/pool-exhaustion")
print("SYSTEM:\n", verification_request.system)
print("USER:\n", verification_request.messages[0].text)
response = client.generate(verification_request)
print("RECORDED RESPONSE:\n", response.text)
verification = VerificationResult.model_validate_json(response.text)
print("PARSED:", verification)
assert decide("Increase DB connection pool ceiling and page the DBA", verification) == "human_approval"

## Step 4: Self-consistency and rule-table control

In [ ]:
samples = []
for index in (1, 2, 3):
    request = next(r for r in build_requests() if r.case_id == f"i06/self-consistency/sample-{index}")
    print("PROMPT:", request.messages[0].text)
    response = client.generate(request)
    parsed = TriageRecommendation.model_validate_json(response.text)
    print("RECORDED RESPONSE:", response.text, "PARSED:", parsed)
    samples.append(parsed.recommended_action)
assert majority_vote(samples) == ("Increase DB connection pool ceiling", 2 / 3)
optimistic = client.generate(next(r for r in build_requests() if r.case_id == "i06/verifier/optimistic"))
assert decide("Restart DB_MAIN to clear the pool", VerificationResult.model_validate_json(optimistic.text)) == "human_approval"

## Takeaway

This replay-backed experiment makes the application control and measured trade-off explicit.

## References

See the course README for the references and further reading.